# Writing to Delta Tables

Reading data is only half the story. The ELT pattern means you're constantly writing — appending new records, overwriting stale partitions, loading files from landing zones, and evolving schemas as upstream systems change.

In this notebook we'll cover:
- `INSERT INTO` and `INSERT OVERWRITE` — the SQL write verbs
- Dynamic partition overwrite — replacing only affected partitions
- `COPY INTO` — idempotent file ingestion from cloud storage
- DataFrame write modes: `append`, `overwrite`, `ignore`, `error`
- Schema enforcement — what Delta rejects and why
- Schema evolution — `mergeSchema` and `overwriteSchema` on write

## INSERT INTO — Append Rows

`INSERT INTO` appends rows to an existing Delta table. The schema must match — extra or missing columns cause an error.

```sql
-- Append specific values
INSERT INTO sales.orders
VALUES ('ORD099', 'CUST01', 250.00, '2024-03-01', 'UK');

-- Append results of a SELECT
INSERT INTO sales.orders
SELECT order_id, customer_id, amount, order_date, region
FROM   staging.new_orders;
```

> `INSERT INTO` is always an **append** — it never removes existing rows. Running it twice on the same source data will duplicate records. Use `MERGE INTO` (covered in notebook 10) when you need upsert semantics.

## INSERT OVERWRITE — Replace All Data

`INSERT OVERWRITE` replaces the entire contents of a table with the result of the query. The old rows are gone; only the new rows remain.

```sql
INSERT OVERWRITE sales.orders_summary
SELECT region, order_date, SUM(amount) AS revenue
FROM   sales.orders
GROUP BY region, order_date;
```

**Key difference from DataFrame `overwrite` mode:**

| | `INSERT OVERWRITE` (SQL) | DataFrame `mode("overwrite")` |
|---|---|---|
| Schema check | Must match existing schema | Can replace schema (with `overwriteSchema=true`) |
| Time travel | Previous versions still accessible | Previous versions still accessible |
| Partition behaviour | Replaces all partitions | By default replaces all partitions; dynamic overwrite replaces only affected ones |

> **Exam tip:** `INSERT OVERWRITE` **cannot change the schema** — it enforces the existing schema strictly. DataFrame `mode("overwrite")` with `.option("overwriteSchema", "true")` can replace the schema entirely.

## Dynamic Partition Overwrite

When a table is partitioned (e.g., by `region`), a full overwrite replaces **all** partitions — even regions not present in the new data. That is often not what you want.

**Dynamic partition overwrite** replaces only the partitions that appear in the incoming data, leaving all other partitions untouched.

```python
# Enable dynamic partition overwrite for this write
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

# Only UK partition will be overwritten; US, DE, FR stay intact
(df_uk_updates
    .write
    .format("delta")
    .mode("overwrite")
    .option("partitionOverwriteMode", "dynamic")
    .saveAsTable("sales.orders"))
```

SQL equivalent using `REPLACE WHERE`:

```sql
-- Atomically replace only the UK partition
INSERT INTO sales.orders
REPLACE WHERE region = 'UK'
SELECT * FROM staging.uk_orders;
```

`REPLACE WHERE` is the preferred Delta Lake pattern — it is explicit, atomic, and does not require setting a session-level config.

## COPY INTO — Idempotent File Ingestion

`COPY INTO` is purpose-built for loading files from cloud storage into a Delta table. Its key property: **idempotency** — it tracks which files have already been loaded and skips them on subsequent runs.

```sql
COPY INTO sales.orders
FROM 's3://my-bucket/landing/orders/'
FILEFORMAT = PARQUET
COPY_OPTIONS ('mergeSchema' = 'true');
```

```sql
-- CSV with format options
COPY INTO sales.raw_orders
FROM (
  SELECT order_id, CAST(amount AS DOUBLE) AS amount, region
  FROM 's3://my-bucket/landing/orders/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'sep' = ',')
COPY_OPTIONS   ('mergeSchema' = 'true');
```

**How COPY INTO tracks state:**
- Loaded file paths are stored in the Delta table's transaction log
- Re-running `COPY INTO` on the same path skips already-loaded files
- New files that arrive after the last run are loaded automatically

**COPY INTO vs Auto Loader:**

| | COPY INTO | Auto Loader |
|---|---|---|
| Trigger | Manual / scheduled job | Continuous or triggered streaming |
| Scale | Good for thousands of files | Good for millions of files (uses cloud file notification) |
| Idempotent | Yes | Yes |
| Schema inference | Yes | Yes (with evolution support) |
| Use case | Batch micro-ingestion | High-volume continuous ingestion |

> **Exam tip:** `COPY INTO` is idempotent and safe to re-run. If new files appear in the source path, the next run picks them up. If no new files exist, it does nothing. This makes it safe to schedule without deduplication logic.

## DataFrame Write Modes

When writing with the PySpark DataFrame API, the `mode` controls what happens if the target already has data:

| Mode | Behaviour |
|------|-----------|
| `append` | Adds rows to the existing table/path. Never removes anything. |
| `overwrite` | Replaces all data. For Delta, previous versions remain accessible via time travel. |
| `ignore` | Does nothing if the target already exists. Silent no-op. |
| `error` *(default)* | Throws an error if the target already exists. |

```python
# Append
df.write.format("delta").mode("append").saveAsTable("sales.orders")

# Full overwrite
df.write.format("delta").mode("overwrite").saveAsTable("sales.orders")

# Overwrite and replace the schema (dangerous — removes old columns)
df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("sales.orders")
```

> `mode("overwrite")` without `overwriteSchema` still enforces the existing schema — it replaces the data but not the column definitions. Use `overwriteSchema=true` only when you intentionally want to change the table structure.

## Schema Enforcement on Write

Delta's schema enforcement rejects writes that don't match the table schema. This is a feature — it protects downstream consumers from silent data corruption.

**What gets rejected:**
```python
# Table schema: order_id STRING, amount DOUBLE, region STRING

# Extra column → AnalysisException: cannot write extra column 'discount'
df_with_extra_col.write.format("delta").mode("append").saveAsTable("sales.orders")

# Wrong type → AnalysisException: cannot write string into double column 'amount'
df_with_wrong_type.write.format("delta").mode("append").saveAsTable("sales.orders")
```

**What is allowed (implicit casting):**
Delta allows safe type widening — e.g., writing an `INT` into a `LONG` column, or `FLOAT` into `DOUBLE`. Narrowing (e.g., `DOUBLE` into `INT`) is rejected.

**Missing columns:**
A DataFrame that is missing columns from the table schema is accepted — missing columns are written as `null`.

## Schema Evolution on Write

When you want to add new columns (not change existing ones), use **schema evolution** instead of fighting schema enforcement.

### `mergeSchema` — Add New Columns

Merges the incoming DataFrame's schema with the existing table schema. New columns in the DataFrame are added to the table; existing columns are unchanged.

```python
# Enable for a single write
df_with_new_col.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("sales.orders")

# Enable for the entire session
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
```

SQL equivalent:
```sql
SET spark.databricks.delta.schema.autoMerge.enabled = true;

INSERT INTO sales.orders
SELECT *, 'new_field_value' AS new_column
FROM staging.orders;
```

### `overwriteSchema` — Replace the Schema Entirely

Drops the existing schema and replaces it with the incoming DataFrame's schema. All data is replaced. Use with caution — this is destructive.

```python
df_new_schema.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("sales.orders")
```

| | `mergeSchema` | `overwriteSchema` |
|---|---|---|
| Adds new columns | Yes | Yes |
| Removes old columns | No | Yes |
| Data preserved | Yes (append or overwrite data, schema widened) | No (full overwrite) |
| Safe for production | Yes | Use with caution |

## Hands-On: Write Patterns

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS write_demo")
spark.sql("USE SCHEMA write_demo")

# Base table with initial data
spark.sql("""
  CREATE OR REPLACE TABLE orders (
    order_id    STRING,
    customer_id STRING,
    amount      DOUBLE,
    region      STRING
  ) USING DELTA
""")

spark.sql("""
  INSERT INTO orders VALUES
    ('ORD001', 'CUST01', 120.50, 'UK'),
    ('ORD002', 'CUST02', 340.00, 'US'),
    ('ORD003', 'CUST01',  89.99, 'UK')
""")

spark.sql("SELECT * FROM orders").show()

In [ ]:
# INSERT INTO — append new rows
spark.sql("""
  INSERT INTO orders VALUES
    ('ORD004', 'CUST03', 210.00, 'DE'),
    ('ORD005', 'CUST02',  55.00, 'US')
""")

print(f"Row count after INSERT INTO: {spark.sql('SELECT COUNT(*) FROM orders').collect()[0][0]}")

In [ ]:
# INSERT OVERWRITE — replace all rows
spark.sql("""
  INSERT OVERWRITE orders VALUES
    ('ORD010', 'CUST99', 999.00, 'FR')
""")

print(f"Row count after INSERT OVERWRITE: {spark.sql('SELECT COUNT(*) FROM orders').collect()[0][0]}")
spark.sql("SELECT * FROM orders").show()

# Previous data still accessible via time travel
print("Version 1 (before overwrite):")
spark.sql("SELECT COUNT(*) AS rows FROM orders VERSION AS OF 1").show()

In [ ]:
# REPLACE WHERE — atomic partition-style overwrite
# Restore some data first
spark.sql("""
  INSERT OVERWRITE orders VALUES
    ('ORD001', 'CUST01', 120.50, 'UK'),
    ('ORD002', 'CUST02', 340.00, 'US'),
    ('ORD003', 'CUST01',  89.99, 'UK'),
    ('ORD004', 'CUST03', 210.00, 'DE')
""")

# Replace only UK rows — US and DE rows are untouched
spark.sql("""
  INSERT INTO orders
  REPLACE WHERE region = 'UK'
  VALUES ('ORD020', 'CUST01', 500.00, 'UK')
""")

spark.sql("SELECT * FROM orders ORDER BY region").show()

In [ ]:
# DataFrame append — equivalent to INSERT INTO
from pyspark.sql import Row

new_rows = spark.createDataFrame([
    Row(order_id="ORD030", customer_id="CUST05", amount=175.0, region="US"),
])

new_rows.write.format("delta").mode("append").saveAsTable("write_demo.orders")
print(f"Row count: {spark.sql('SELECT COUNT(*) FROM orders').collect()[0][0]}")

In [ ]:
# Schema enforcement — extra column is rejected
df_extra = spark.createDataFrame([
    Row(order_id="ORD099", customer_id="CUST01", amount=10.0, region="UK", discount=5.0)
])

try:
    df_extra.write.format("delta").mode("append").saveAsTable("write_demo.orders")
except Exception as e:
    print(f"Schema enforcement caught: {type(e).__name__}")
    print(str(e)[:200])

In [ ]:
# Schema evolution — mergeSchema adds the new column
df_extra.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("write_demo.orders")

# Table now has 'discount' column; existing rows have null there
spark.sql("DESCRIBE orders").show()
spark.sql("SELECT * FROM orders WHERE discount IS NOT NULL").show()

In [ ]:
# Inspect write history
spark.sql("DESCRIBE HISTORY orders").select(
    "version", "timestamp", "operation", "operationParameters"
).show(10, truncate=False)

In [ ]:
# Cleanup
spark.sql("DROP SCHEMA IF EXISTS write_demo CASCADE")

## Decision Guide

```
I want to...                                          → Use
────────────────────────────────────────────────────────────────
Append new rows to an existing table                  → INSERT INTO / mode("append")
Replace all rows but keep the schema                  → INSERT OVERWRITE / mode("overwrite")
Replace only rows matching a condition                → INSERT INTO ... REPLACE WHERE
Replace only partitions present in incoming data      → mode("overwrite") + partitionOverwriteMode=dynamic
Replace schema AND data completely                    → mode("overwrite") + overwriteSchema=true
Load new files from cloud storage, skip already-loaded → COPY INTO
Add new columns from incoming data                    → mode("append") + mergeSchema=true
Upsert (insert new, update existing)                  → MERGE INTO  (notebook 10)
```

## Summary

- `INSERT INTO` appends rows — running it twice duplicates data. Use `MERGE INTO` for upserts.
- `INSERT OVERWRITE` replaces all rows but cannot change the table schema.
- `INSERT INTO ... REPLACE WHERE` atomically replaces rows matching a predicate — the clean alternative to dynamic partition overwrite.
- `COPY INTO` loads files from cloud storage idempotently — it tracks loaded files in the Delta log and skips them on re-runs.
- DataFrame write modes: `append`, `overwrite`, `ignore`, `error` (default).
- Delta's **schema enforcement** rejects extra columns and type mismatches. Missing columns are written as null.
- `mergeSchema=true` adds new columns from the incoming data without removing existing ones. `overwriteSchema=true` replaces the schema entirely — use with care.

Next: `06-cleaning-transforming-data.ipynb` — filtering, casting, deduplication, and null handling in the Silver layer.